<style>
.jp-RenderedHTMLCommon { color: #EAEAEA; }
.manus-title {
  background: linear-gradient(135deg, #111827, #1F2937);
  border: 1px solid #374151;
  border-radius: 18px;
  padding: 22px 26px;
  margin: 10px 0 18px 0;
  box-shadow: 0 8px 28px rgba(0,0,0,.28);
}
.manus-title h1 { color: #F9FAFB; margin: 0; font-size: 30px; }
.manus-title p { color: #D1D5DB; margin: 8px 0 0 0; font-size: 15px; }
.callout {
  border-radius: 14px;
  padding: 14px 16px;
  margin: 14px 0;
  border-left: 5px solid;
}
.info { border-color: #60A5FA; background: #0F172A; }
.good { border-color: #34D399; background: #052E2B; }
.bad { border-color: #F87171; background: #2A1010; }
.warn { border-color: #FBBF24; background: #2A2108; }
.key { border-color: #A78BFA; background: #1E1633; }
table { width: 100%; border-collapse: collapse; font-size: 14px; }
th { background: #1F2937; color: #F9FAFB; padding: 10px; border: 1px solid #374151; }
td { padding: 10px; border: 1px solid #374151; vertical-align: top; }
tr:nth-child(even) td { background: #111827; }
tr:nth-child(odd) td { background: #0B1120; }
code { color: #FDE68A; background: #1F2937; padding: 2px 5px; border-radius: 6px; }
pre { background: #0B1020 !important; border: 1px solid #374151; border-radius: 12px; padding: 12px; }
</style>

<div class="manus-title">
  <h1>🧠 Manus — AI Agent Context Engineering（中文版）</h1>
  <p>深色风格工程笔记：适用于 LangGraph、Playwright bug-fix agent、Career Radar、MCP 和 production agent 系统。</p>
</div>

## 🎯 核心思想

<div class="callout key">
<strong>Production Agent ≠ LLM + Tools</strong><br>
真正的 Production Agent = <strong>模型 + Context 架构 + Memory + Error 反馈 + Attention 控制</strong>
</div>

这篇 Manus 文章真正想表达的是：

> Agent 的上限，不只是模型能力决定的，更取决于你如何设计它每一步看到什么、记住什么、忘掉什么、如何使用工具、如何从错误中恢复。

换句话说，**context engineering 是 production agent 的核心工程能力**。

## 🗺️ 一页总览

| # | 原则 | 核心问题 | Manus 解决方案 | 对你的意义 |
|---|---|---|---|---|
| 1 | ⚡ KV Cache First | context 前缀乱变 → 慢 + 贵 | Stable Prefix + Append-only + deterministic serialization | system prompt 不要动态变化 |
| 2 | 🧰 Mask, Don’t Remove | 动态删 tools → cache miss + hallucinated tool | tool list 永远固定，只限制当前可选范围 | `ALL_TOOLS` 固定，state 控制 allowed tools |
| 3 | 📁 File System = Memory | logs / DOM / PDF / diff 把 context 撑爆 | 大内容写文件，context 只保留路径 | `logs/ screenshots/ corpus/ diffs/` |
| 4 | 📝 Recitation | 长任务容易忘目标、跑偏 | 持续维护并重读 `todo.md` | LangGraph 每个 major node 更新 TODO |
| 5 | 🧯 Keep Errors | 清理错误 → 模型学不到东西 | 保留失败 action、stack trace、tool error | bug-fix automation 必须保留错误 |
| 6 | 🔁 Don’t Few-Shot Yourself | pattern 太重复 → agent autopilot | 加入小的结构化变化 | Career Radar 批量分析时很重要 |

## 1. ⚡ Design Around KV Cache

<div class="callout info">
<strong>Agent 每一步都在重复读前面的 context。</strong><br>
如果前缀不稳定，cache 就失效，成本和延迟都会上升。
</div>

Agent loop 通常是：

```text
User Task
→ Model decides action
→ Tool runs
→ Observation returns
→ Append action + observation to context
→ Repeat
```

### ❌ 错误做法

```python
system_prompt = f"""
You are a bug-fixing agent.

Current time: {datetime.now()}
Current branch: {branch}
Current memory: {memory}
"""
```

问题：

- 每一步 `time` 都变，prefix 不稳定。
- prefix 变了，KV cache 很容易失效。
- agent loop 越长，成本越高，延迟越明显。

### ✅ 正确做法

```python
SYSTEM_PROMPT = "You are a production bug-fixing agent."

messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": f"Current time: {now}. Current branch: {branch}."}
]
```

<div class="callout good">
<strong>口诀：</strong><br>
Prefix 不动，钱不白烧。
</div>

| Bad | Good |
|---|---|
| `system=f"time={now}"` | fixed `SYSTEM_PROMPT` |
| 重写历史 messages | 只 append 新 message |
| JSON key 顺序随机 | deterministic serialization |
| 每个 node 改 tool schema | fixed tool registry |

## 2. 🧰 Mask, Don’t Remove

<div class="callout warn">
<strong>这是全文最值钱的一条之一。</strong><br>
不要在 agent 每一步动态删除工具。tool list 最好保持稳定，只限制当前允许选择哪些工具。
</div>

### 假设你的 tools

```python
ALL_TOOLS = [
    browser_open,
    browser_click,
    browser_screenshot,
    shell_run,
    file_read,
    file_write,
    git_diff,
    git_commit,
    slack_send,
]
```

### ❌ 错误做法：Remove Tools

```python
if state == "browser":
    tools = [browser_open, browser_click, browser_screenshot]

if state == "shell":
    tools = [shell_run]
```

问题：

1. tool definitions 通常在 context 前面，tool schema 一变，KV cache 就容易失效。
2. 历史 action 里可能有 `browser_open`，但当前 tools 里它消失了。
3. 模型会困惑，可能 hallucinate 一个不存在的 tool。

例子：

```text
History:
Action: browser_open

Current tools:
shell_run
file_read

Model 可能输出:
browser_open_url
```

`browser_open_url` 根本不存在。

### ✅ 正确做法：Mask Tools

```python
ALL_TOOLS = [
    browser_open,
    browser_click,
    browser_screenshot,
    shell_run,
    file_read,
    file_write,
    git_diff,
    git_commit,
    slack_send,
]

state_policy = {
    "browser": ["browser_"],
    "shell": ["shell_"],
    "editing": ["file_"],
    "git": ["git_"],
    "notify": ["slack_"],
}
```

<div class="callout key">
<strong>心理模型：</strong><br>
Remove = 每一步都改菜单。<br>
Mask = 菜单不变，但当前只能点某一类菜。
</div>

### Tool 命名很重要

| Good Tool Names | Bad Tool Names |
|---|---|
| `browser_open` | `open` |
| `browser_click` | `clickThing` |
| `shell_run` | `execute` |
| `file_write` | `save` |
| `git_commit` | `commitCode` |

统一 prefix 可以让 state-based routing 很容易。

## 3. 📁 File System = Long-Term Memory

<div class="callout info">
<strong>真实 agent 会产生大量 observation。</strong><br>
比如网页内容、PDF、DOM snapshot、console logs、截图、test output、git diff。
</div>

### ❌ 错误做法：全部塞进 context

```text
500 lines console logs
3000 lines DOM snapshot
1000 lines test output
800 lines git diff
OCR text
PDF text
...
```

后果：

- token 爆炸
- 很贵
- 很慢
- 超 context window
- 模型在长 context 中容易 lost in the middle

### ✅ 正确做法：写入文件系统

```python
write_file("logs/console_error.log", console_output)
write_file("debug/dom_snapshot.html", dom_snapshot)
write_file("screenshots/login_failure.png", screenshot)
write_file("patches/current_diff.patch", git_diff)
```

context 只保留引用：

```text
Console log saved to logs/console_error.log
DOM snapshot saved to debug/dom_snapshot.html
Screenshot saved to screenshots/login_failure.png
```

需要时再读：

```python
read_file("logs/console_error.log")
```

<div class="callout good">
<strong>口诀：</strong><br>
Memory ≠ Tokens。Memory = Filesystem + Paths + Selective Retrieval。
</div>

| Project | File Memory Design |
|---|---|
| Playwright bug-fix agent | `logs/`, `screenshots/`, `dom/`, `diffs/`, `test_results/` |
| AI Career Radar | `corpus/company_role_date/job.md`, `metadata.json`, `analysis.md` |
| Email agent | `threads/`, `inbound/`, `outbound/`, `errors/` |
| LangGraph experiments | `runs/`, `checkpoints/`, `observations/`, `todos/` |

## 4. 📝 Recitation = Attention Control

<div class="callout warn">
<strong>长任务会跑偏。</strong><br>
任务目标一开始出现，但几十步之后，它可能被埋在 context 中间。
</div>

### ❌ 问题

```text
Step 0:
Goal = fix login regression

Step 30:
Agent is reading package.json and may forget the original goal.
```

### ✅ Manus 做法：维护 todo.md

```markdown
# TODO

[x] Reproduce the login bug
[x] Capture screenshot
[x] Collect console errors
[ ] Identify failing component
[ ] Patch the code
[ ] Run tests
[ ] Generate final summary
```

每隔几步更新并重读：

```python
update_file("todo.md", new_todo)
read_file("todo.md")
```

<div class="callout key">
<strong>为什么有效：</strong><br>
模型更容易注意最近的 tokens。不断重写 TODO，相当于把目标重新放到 context 尾部。
</div>

### LangGraph State 示例

```python
class AgentState(TypedDict):
    task: str
    todo_path: str
    current_step: str
    artifacts: list[str]
    errors: list[str]
```

<div class="callout good">
<strong>口诀：</strong><br>
Todo 不是 UI，是 attention hack。
</div>

## 5. 🧯 Keep the Wrong Stuff In

<div class="callout bad">
<strong>坏习惯：</strong><br>
工具失败后清理 trace，然后 silent retry。
</div>

### ❌ 错误做法

```python
try:
    run_tests()
except Exception:
    clear_history()
    retry()
```

问题：

- 模型看不到失败原因。
- 它可能重复犯同一个错误。
- 系统丢失了关键 debugging evidence。

### ✅ 正确做法

保留失败 action 和 observation：

```text
Action:
shell_run("npm test")

Observation:
ModuleNotFoundError: Cannot find module '@playwright/test'
```

下一步会更自然：

```python
shell_run("npm install -D @playwright/test")
```

<div class="callout good">
<strong>口诀：</strong><br>
Failures are not noise. Failures are learning signals.
</div>

| Failure Type | Keep It Where |
|---|---|
| Stack trace | `logs/stack_trace.log` |
| Test failure | `test_results/failure.txt` |
| Browser console error | `logs/browser_console.log` |
| Timeout | `logs/timeouts.log` |
| Bad tool call | context history + `errors/tool_error.json` |

## 6. 🔁 Don’t Few-Shot Yourself

<div class="callout info">
<strong>LLM 会模仿 context 里的模式。</strong><br>
长 workflow 中，agent 自己的历史会变成 accidental few-shot examples。
</div>

### ❌ 坏模式：过度重复

```text
Open file
Extract skills
Save summary

Open file
Extract skills
Save summary

Open file
Extract skills
Save summary
```

后果：

- 模型开始 autopilot
- 不认真看内容
- overgeneralization
- hallucination
- summary 变机械

### ✅ 好做法：加入小的结构化变化

```text
Opened: meta_job.md
Loaded: anthropic_role.md
Reading: openai_position.md
Parsing: google_job.md
```

可以变化：

- observation wording
- ordering
- summary template
- formatting

<div class="callout key">
<strong>口诀：</strong><br>
太整齐，agent 反而更脆弱。
</div>

### Career Radar 里的做法

schema 保持稳定：

```json
{
  "company": "...",
  "role": "...",
  "skills": [],
  "agent_relevance": "...",
  "notes_path": "..."
}
```

但自然语言 observation 可以有轻微变化。

## 🧩 对你的项目的直接映射

| Your Project | Manus Principle | Concrete Implementation |
|---|---|---|
| LangGraph bug-fix agent | Mask, don't remove | Global `ALL_TOOLS`; state decides allowed prefixes |
| Playwright automation | File system memory | Save screenshots, DOM, console logs, test outputs as files |
| AI Career Radar | External memory | `corpus/` folder with job post, metadata, extracted skills |
| Email agent | Keep errors | Preserve SendGrid/webhook/API failures in logs |
| Long-running agent flow | Recitation | Maintain `todo.md` throughout the graph |
| MCP tool ecosystem | Fixed tool schema | Avoid constantly changing exposed tool list mid-run |

## ✅ Implementation Checklist

| Check | Question |
|---|---|
| ⬜ Stable prefix | system prompt 是否跨步骤保持固定？ |
| ⬜ Append-only context | 是否避免重写旧历史？ |
| ⬜ Deterministic serialization | JSON key/order 是否稳定？ |
| ⬜ Fixed tool registry | 是否避免动态删除 tool？ |
| ⬜ Tool masking | state 是否只限制可选工具，而不是改 schema？ |
| ⬜ File memory | 大输出是否写入文件？ |
| ⬜ Restorable compression | 如果从 context 删除内容，是否能通过 path/URL 恢复？ |
| ⬜ TODO recitation | agent 是否周期性刷新目标？ |
| ⬜ Error retention | 失败信息是否保留下来帮助自我修正？ |
| ⬜ Anti-autopilot variation | 是否避免过度重复的 trace？ |

## 🧠 最终记忆钩子

| Hook | 含义 |
|---|---|
| ⚡ Stable Prefix = Cheap Agent | 前缀稳定，cache 才有效 |
| 🧰 Fixed Tools = Stable Agent | tool schema 不要每步乱变 |
| 📁 Filesystem > Long Context | 大 observation 应该存在模型窗口外 |
| 📝 TODO = Attention Control | 把目标不断放回 context 尾部 |
| 🧯 Failures = Learning Signals | 错误帮助 agent 避免重复犯错 |
| 🔁 Uniform Context = Fragile Agent | 太重复会让 agent 机械化 |